# 00 — Refinement foundations

Freeze the measurement problem **before** solving it. Stage 10 remains the confirmatory
taxonomy baseline; Stage 11 is post-hoc measurement correction.

This notebook documents source call49 mappings, lookup-derived candidate pools,
Stage 10 δ freeze, known measurement failures, blinding, and integrity traps
(H2 pool 10≠11; leaf `7.2` = 12≠13).

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = cwd
for _ in range(6):
    if (root / "configs").is_dir() and (root / "src").is_dir():
        break
    root = root.parent
else:
    raise RuntimeError("project root not found")
sys.path.insert(0, str(root))

from src.stage11_refined_construct_analysis.analysis import notebook_helpers as nh
from src.stage11_refined_construct_analysis.analysis import review_display as rd
from src.stage11_refined_construct_analysis.lookup import (
    load_topic_lookup,
    run_lookup_integrity,
    topics_for_leaves,
)

ctx = nh.setup("00_refinement_foundations")
cfg = ctx.cfg

Project root : /home/polina/Documents/Cursor_Projects/romantic_novels_large_corpus
Config       : configs/stage11/refined_constructs.yaml
Run          : v4_l12_granular_final_call49
Outputs      : results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations


## 1. Frozen inputs and integrity

In [2]:
lookup = load_topic_lookup(cfg)
integrity = run_lookup_integrity(cfg, lookup)
integrity.raise_if_failed()
display(pd.DataFrame(integrity.checks))

frozen = nh.load_frozen_inputs(cfg)
print("frozen_inputs keys:", sorted(frozen.keys()) if frozen else "(missing — run 01_build_candidate_manifests.py)")
if frozen:
    ctx.save_markdown(json.dumps(frozen, indent=2), "frozen_inputs_snapshot")

,name,ok,detail,topic_ids,by_leaf,n_topics,unmeasurable
0,h2_pool_size,True,"H2 pool leaves ['4.5', '5.3a', '8.3a'] yield 1...","[29, 61, 62, 65, 128, 157, 167, 204, 242, 305]","{'4.5': [29, 62, 65, 128, 157, 204, 242, 305],...",NaN,NaN
1,h2_leaf_4.5,True,"leaf 4.5: n=8, ids=[29, 62, 65, 128, 157, 204,...","[29, 62, 65, 128, 157, 204, 242, 305]",NaN,8.0000,NaN
2,h2_leaf_5.3a,True,"leaf 5.3a: n=1, ids=[167]",[167],NaN,1.0000,NaN
3,h2_leaf_8.3a,True,"leaf 8.3a: n=1, ids=[61]",[61],NaN,1.0000,NaN
4,leaf_7_2_count,True,"leaf 7.2 has 12 topics (expected 12); ids=[51,...","[51, 78, 82, 87, 113, 114, 117, 148, 249, 269,...",NaN,12.0000,NaN
5,empty_leaf_2.4,True,leaf 2.4 should be empty; found 0 topics [],NaN,NaN,0.0000,True
6,empty_leaf_6.1a,True,leaf 6.1a should be empty; found 0 topics [],NaN,NaN,0.0000,True
7,empty_leaf_6.7,True,leaf 6.7 should be empty; found 0 topics [],NaN,NaN,0.0000,True
8,topic_91_leaf,True,topic 91 taxonomy_main_id=7.1 (expected 7.1),NaN,NaN,NaN,NaN


frozen_inputs keys: ['empty_leaves', 'exhaustive_topic_ids', 'h2_n', 'h2_topic_ids', 'integrity_ok', 'n_audited_topics', 'run_id', 'topic_lookup']
  saved markdown: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations/tables/frozen_inputs_snapshot.md


## 2. Stage 10 δ freeze (confirmatory baseline — do not overwrite)

In [3]:
delta_freeze = cfg.section("stage10_delta_freeze")
delta_tbl = pd.DataFrame(
    [{"hypothesis": k, "stage10_cliffs_delta": v} for k, v in delta_freeze.items()]
)
display(delta_tbl)
ctx.save_table(delta_tbl, "stage10_delta_freeze")

,hypothesis,stage10_cliffs_delta
0,H1,-0.0290
1,H2,0.0270
2,H3,-0.1460
3,H4,0.0900
4,H5,0.0120
5,H6,0.0440


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations/tables/stage10_delta_freeze.csv  (6 rows)


## 3. Measurement-problem table

In [4]:
problems = pd.DataFrame(
    [
        {
            "construct_targeted": "explicit sex",
            "old_operationalisation": "2.3 — Explicit Sexual Acts",
            "known_problem": "only ~28% genuinely explicit; kissing/undressing dominate",
            "new_audit": "H1 intimacy",
        },
        {
            "construct_targeted": "HEA / final payoff",
            "old_operationalisation": "4.5 + thin 5.3a + 8.3a",
            "known_problem": "confession/repair ≠ final payoff; thin leaves (1 topic each)",
            "new_audit": "H2 HEA",
        },
        {
            "construct_targeted": "material/social display",
            "old_operationalisation": "1.6 + 8.2 + 5.3a + 8.3a",
            "known_problem": "appearance ≠ material security; 1.6 effect robust on-label",
            "new_audit": "H3 security",
        },
        {
            "construct_targeted": "protectiveness",
            "old_operationalisation": "all 4.6 − 4.7",
            "known_problem": "4.6 mixes reassurance/medical/institutional; 4.7 only 2 topics",
            "new_audit": "H4 protection",
        },
        {
            "construct_targeted": "darkness / tenderness",
            "old_operationalisation": "broad AX_dark_vs_tender",
            "known_problem": "pools relational conflict, violence, external danger, affect",
            "new_audit": "H5 darkness",
        },
        {
            "construct_targeted": "narrative arc",
            "old_operationalisation": "tertile rising − falling leaves",
            "known_problem": "conflict/secrecy may not be main-couple",
            "new_audit": "H6 arc",
        },
    ]
)
display(problems)
ctx.save_table(problems, "measurement_problem_table")

,construct_targeted,old_operationalisation,known_problem,new_audit
0,explicit sex,2.3 — Explicit Sexual Acts,only ~28% genuinely explicit; kissing/undressi...,H1 intimacy
1,HEA / final payoff,4.5 + thin 5.3a + 8.3a,confession/repair ≠ final payoff; thin leaves ...,H2 HEA
2,material/social display,1.6 + 8.2 + 5.3a + 8.3a,appearance ≠ material security; 1.6 effect rob...,H3 security
3,protectiveness,all 4.6 − 4.7,4.6 mixes reassurance/medical/institutional; 4...,H4 protection
4,darkness / tenderness,broad AX_dark_vs_tender,"pools relational conflict, violence, external ...",H5 darkness
5,narrative arc,tertile rising − falling leaves,conflict/secrecy may not be main-couple,H6 arc


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations/tables/measurement_problem_table.csv  (6 rows)


## 4. Candidate pools (lookup-derived — never hard-coded counts)

In [5]:
rows = []
for hyp in ("H1", "H2", "H3", "H4", "H5", "H6"):
    cand = nh.load_candidates(cfg, hyp)
    n = int(cand.get("n_topics") or len(cand.get("topic_ids") or []))
    rows.append(
        {
            "hypothesis": hyp,
            "n_topics": n,
            "name": cand.get("name"),
        }
    )
pool_summary = pd.DataFrame(rows)
display(pool_summary)
ctx.save_table(pool_summary, "candidate_pool_summary")

# H2 explicit assert — print id — label, not bare ids
h2_ids = topics_for_leaves(lookup, ["4.5", "5.3a", "8.3a"])
expected = int(cfg.section("integrity", "h2_expected_n_topics"))
print(f"H2 pool from lookup: {len(h2_ids)} topics (expected {expected})")
assert len(h2_ids) == expected, f"H2 pool size {len(h2_ids)} != {expected}"
print("H2 pool (id — label):")
for line in rd.labeled_topic_list(lookup, h2_ids):
    print(f"  · {line}")

ids_72 = topics_for_leaves(lookup, ["7.2"])
n_72 = len(ids_72)
assert n_72 == int(cfg.section("integrity", "leaf_7_2_expected_n"))
print(f"Leaf 7.2: {n_72} topics (ok). Sample:")
for line in rd.labeled_topic_list(lookup, ids_72[:8]):
    print(f"  · {line}")
if n_72 > 8:
    print(f"  … and {n_72 - 8} more")

,hypothesis,n_topics,name
0,H1,90,intimacy
1,H2,10,hea_payoff
2,H3,97,security_material
3,H4,32,protection_possession
4,H5,22,darkness_tenderness
5,H6,28,arc_semantics


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations/tables/candidate_pool_summary.csv  (6 rows)
H2 pool from lookup: 10 topics (expected 10)
H2 pool (id — label):
  · 29 — Confessing Long-Held Love
  · 61 — Planning to Exchange Rings
  · 62 — Admitting You've Been Stupid
  · 65 — Declaring A True Partnership
  · 128 — Confessing How Much You've Missed
  · 157 — Swearing to Save Him From Himself
  · 167 — Planning A Wedding Reception
  · 204 — Promising to Care For Her Sister
  · 242 — Trading Forgiveness For Old Wrongs
  · 305 — Confessing A Lifelong Regret
Leaf 7.2: 12 topics (ok). Sample:
  · 51 — Locked Up By A Vampire
  · 78 — Swearing War Before He Takes Her
  · 82 — Touch Her and Your Family Suffers
  · 87 — Threatening Death As A Warning
  · 113 — Knife Handed Over For Combat
  · 114 — Guns Aimed Across The Room
  · 117 — Blamed and Threatened Into Compliance
  · 148 — Banished From The Ranch
  … and 

## 5. Blinding and evidence completeness

In [6]:
cell_key = nh.load_cell_key(cfg)
print("Cell key sealed until notebook 10. Labels:", list(cell_key.get("labels", cell_key.keys()))[:8])

pkt_dir = cfg.output_path("evidence_packets_dir")
n_packets = len(list(pkt_dir.glob("topic_*.json"))) if pkt_dir.exists() else 0
print(f"Evidence packets on disk: {n_packets}")

audit_counts = []
for hyp in ("H1", "H2", "H3", "H4", "H5", "H6"):
    for pass_name in ("A", "B", "C"):
        df = nh.load_audit_jsonl(cfg, hyp, pass_name)
        audit_counts.append({"hypothesis": hyp, "pass": pass_name, "n": len(df)})
audit_cov = pd.DataFrame(audit_counts)
display(audit_cov.pivot(index="hypothesis", columns="pass", values="n"))
ctx.save_table(audit_cov, "audit_completeness")

print("\nFoundations frozen. Proceed to hypothesis audits 01–06 (no rating peek).")

Cell key sealed until notebook 10. Labels: ['CELL_A', 'CELL_B', 'CELL_C', 'CELL_D']
Evidence packets on disk: 174


pass,A,B,C
hypothesis,,,
H1,98,98,98
H2,10,10,10
H3,82,82,82
H4,32,32,32
H5,22,22,22
H6,29,29,29


  saved table: results/stage11_refined_construct_analysis/v4_l12_granular_final_call49/notebook_analysis/00_refinement_foundations/tables/audit_completeness.csv  (18 rows)

Foundations frozen. Proceed to hypothesis audits 01–06 (no rating peek).
